In [0]:
create schema if not exists demo_db

In [0]:
--In Databricks create schema should be fine. Create database is Spark specific feature
--create database if not exists demo_db

In [0]:
--  Create fire service table in demo_db where the table will be loaded from the fire service csv file
use demo_db;
create table if not exists fire_service_calls_tbl(
  CallNumber integer,
  UnitID string,
  IncidentNumber integer,
  CallType string,
  CallDate string,
  WatchDate string,
  CallFinalDisposition string,
  AvailableDtTm string,
  Address string,
  City string,
  Zipcode integer,
  Battalion string,
  StationArea string,
  Box string,
  OriginalPriority string,
  Priority string,
  FinalPriority integer,
  ALSUnit boolean,
  CallTypeGroup string,
  NumAlarms integer,
  UnitType string,
  UnitSequenceInCallDispatch integer,
  FirePreventionDistrict string,
  SupervisorDistrict string,
  Neighborhood string,
  Location string,
  RowID string,
  Delay float
) ;

In [0]:
--Load fire service csv file into fire_service_calls_tbl table
create or replace temporary view fire_service_calls_vw
using csv
options (
  Path "/databricks-datasets/learning-spark-v2/sf-fire/sf-fire-calls.csv",
  header = "true",
  inferSchema = "true",
  delimiter = ','
);


In [0]:
insert into fire_service_calls_tbl
select * from fire_service_calls_vw;

In [0]:
select * from demo_db.fire_service_calls_tbl;

In [0]:
select * from fire_service_calls_vw;

In [0]:
-- Caching spark table data in lazy manner
cache lazy table fire_service_calls_tbl_cache as
select * from demo_db.fire_service_calls_tbl;

-- Error: CACHE TABLE AS SELECT is not supported on serverless compute.

#####Q1. How many distinct types of calls were made to the Fire Department?

In [0]:
select CallType, count(1) 
from demo_db.fire_service_calls_tbl
group by CallType;

##### Q2. Find out all response for delayed times greater than 5 mins?

In [0]:
select CallNumber, CallType, CallDate, Delay from demo_db.fire_service_calls_tbl
where Delay > 5

##### Q3. What were the top 5 common call types?

In [0]:
Select CallType, CallCount
from (
  select CallType, count(1) CallCount
  from demo_db.fire_service_calls_tbl
  where callType is not null
  group by CallType
) t
order by t.callcount desc
limit 5

##### Q4. What San Francisco neighborhoods are in the zip codes 94102 and 94103?

In [0]:
select distinct Neighborhood, Zipcode
from demo_db.fire_service_calls_tbl
where zipcode in (94102, 94103)

#####Q5. What was the sum of all call alarms, average, min, and max of the call response times?

In [0]:
select sum(NumAlarms), avg(Delay), min(Delay), max(Delay) 
from demo_db.fire_service_calls_tbl

##### Q6. How many distinct years of data is in the data set?


In [0]:
select distinct year(to_date(CallDate, 'yyyy-MM-dd')) as Unique_Years 
from demo_db.fire_service_calls_tbl;

##### Q7. What week of the year in 2018 had the most fire calls?

In [0]:
select weekofyear(to_date(CallDate, 'yyyy-MM-dd')) as week_of_year, count(1) as CalL_Count
from demo_db.fire_service_calls_tbl
where year(to_date(CallDate, 'yyyy-MM-dd')) = 2018
group by week_of_year
order by CalL_Count desc
limit 1;

##### Q8. What neighborhoods in San Francisco had the worst response time in 2018?

In [0]:
select Neighborhood, max(Delay) as Max_Delay
from demo_db.fire_service_calls_tbl
where City = 'San Francisco'
and year(to_date(CallDate, 'yyyy-MM-dd')) = 2018
group by Neighborhood
order by Max_Delay desc
limit 1;